# 03 — SHAP Explanations

For each window pair (A, B), the notebook computes per-replica SHAP attributions on
the flagged set F_{A,B}. The explainer used depends on `MODEL_TYPE`:

| MODEL_TYPE  | Explainer               | Properties                                    |
|-------------|-------------------------|-----------------------------------------------|
| `xgboost`   | `shap.TreeExplainer`    | deterministic, exact; runs on CPU             |
| `logreg`    | — (skipped)             | coefficients are the explanation              |
| `mlp_plr`   | `shap.GradientExplainer`| stochastic, approximate; requires GPU         |

**Output layout** (under `data/shap/{MODEL_TYPE}/pair_{pid:02d}/`):
- `shap_A.npy`, `shap_B.npy` — shape `(R, |F|, p)`, float32
- `expected_values_A.npy`, `expected_values_B.npy` — shape `(R,)`, float32
- `stochasticity.json` — `{best_replica_A, max_abs_diff, median_abs_diff*, is_deterministic}`
  *(\*`median_abs_diff` only present for MLP-PLR)*
- `global_importance_pair00_A.png` — top-20 global importance plot for pair 0

**Coordinate system note (MLP-PLR):** MLP-PLR models are trained on *scaled* features
(each replica has its own `StandardScaler`). SHAP attributions are computed in the
scaled-feature space and saved as such. Feature positions still match the column
ordering in `feature_names.json` — only magnitudes differ. Cosine distance and
RBO-on-|attribution| rankings in notebook 04 are robust to per-replica scaler differences.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import shap
from pathlib import Path

WORKSPACE = Path('/content/drive/MyDrive/HomeCredit_workspace')
PROC_DIR  = WORKSPACE / 'data' / 'processed'
WIN_DIR   = WORKSPACE / 'data' / 'windows'

# ── Model type — must match the MODEL_TYPE used in notebook 02 / 02b ──
MODEL_TYPE = 'logreg'   # 'xgboost' | 'logreg' | 'mlp_plr'

MODEL_DIR = WORKSPACE / 'data' / 'models' / MODEL_TYPE
SHAP_DIR  = WORKSPACE / 'data' / 'shap'  / MODEL_TYPE
SHAP_DIR.mkdir(parents=True, exist_ok=True)

print(f'SHAP version: {shap.__version__}')
print(f'MODEL_TYPE : {MODEL_TYPE}')

In [ ]:
# Load data and window config
X      = pd.read_parquet(PROC_DIR / 'X.parquet').values.astype(np.float32)
with open(PROC_DIR / 'feature_names.json') as f:
    feature_names_json = json.load(f)
feature_names = feature_names_json['all']

with open(WIN_DIR / 'window_config.json') as f:
    config = json.load(f)

R     = config['parameters']['R']
pairs = config['pairs']

print(f'X: {X.shape}, features: {len(feature_names)}')
print(f'R={R}, {len(pairs)} pairs')

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# MLP-PLR setup — only runs when MODEL_TYPE == 'mlp_plr'.
# The MLPPLR / PLREmbedding classes below must be byte-identical with the copies
# in 02b_training_replicas_mlp_plr.ipynb.
# ═════════════════════════════════════════════════════════════════════════════

if MODEL_TYPE == 'mlp_plr':
    import math
    import torch
    import torch.nn as nn

    assert torch.cuda.is_available(), 'MLP-PLR SHAP requires a GPU runtime.'
    DEVICE = torch.device('cuda')
    print(f'PyTorch : {torch.__version__}')
    print(f'Device  : {torch.cuda.get_device_name(0)}')

    num_col_idx = [feature_names.index(fn) for fn in feature_names_json['num']]
    bin_col_idx = [i for i in range(len(feature_names))
                   if i not in set(num_col_idx)]
    n_num = len(num_col_idx)
    n_bin = len(bin_col_idx)
    print(f'Numeric cols: {n_num}  Binary cols: {n_bin}')


    class PLREmbedding(nn.Module):
        def __init__(self, n_num_features: int, k_periodic: int,
                     sigma_init: float, d_embedding: int):
            super().__init__()
            self.n_num_features = n_num_features
            self.k_periodic     = k_periodic
            self.d_embedding    = d_embedding
            self.c = nn.Parameter(torch.randn(n_num_features, k_periodic) * sigma_init)
            self.W = nn.Parameter(torch.empty(n_num_features, 2 * k_periodic, d_embedding))
            self.b = nn.Parameter(torch.zeros(n_num_features, d_embedding))
            nn.init.kaiming_uniform_(self.W, a=math.sqrt(5))

        def forward(self, x_num: torch.Tensor) -> torch.Tensor:
            v = (2.0 * math.pi) * self.c.unsqueeze(0) * x_num.unsqueeze(-1)
            pe = torch.cat([torch.sin(v), torch.cos(v)], dim=-1)
            out = torch.einsum('bnk,nkd->bnd', pe, self.W) + self.b.unsqueeze(0)
            out = torch.relu(out)
            return out.reshape(out.shape[0], -1)


    class MLPPLR(nn.Module):
        def __init__(self, n_num_features: int, n_bin_features: int,
                     k_periodic: int, sigma_init: float, d_embedding: int,
                     n_layers: int, hidden_dim: int, dropout: float,
                     num_col_idx, bin_col_idx):
            super().__init__()
            self.n_num_features = n_num_features
            self.n_bin_features = n_bin_features
            self.register_buffer('num_col_idx_buf',
                                 torch.as_tensor(num_col_idx, dtype=torch.long))
            self.register_buffer('bin_col_idx_buf',
                                 torch.as_tensor(bin_col_idx, dtype=torch.long))
            self.plr = PLREmbedding(n_num_features, k_periodic, sigma_init, d_embedding)
            input_dim = n_num_features * d_embedding + n_bin_features
            layers = []
            in_dim = input_dim
            for _ in range(n_layers):
                layers.extend([nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(dropout)])
                in_dim = hidden_dim
            self.backbone = nn.Sequential(*layers)
            self.head     = nn.Linear(hidden_dim, 1)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            x_num = x.index_select(1, self.num_col_idx_buf)
            x_bin = x.index_select(1, self.bin_col_idx_buf)
            emb   = self.plr(x_num)
            z     = torch.cat([emb, x_bin], dim=-1)
            h     = self.backbone(z)
            return self.head(h).squeeze(-1)


    class ShapOutputAdapter(nn.Module):
        """Wrap an MLPPLR so its output is 2-D (batch, 1) instead of 1-D.

        `shap.GradientExplainer` indexes outputs with `outputs[:, idx]` internally,
        which raises `IndexError` on a bare MLPPLR (whose forward ends with .squeeze(-1)
        for BCEWithLogitsLoss). This adapter preserves gradients and adds back the
        trailing dimension.
        """
        def __init__(self, inner: nn.Module):
            super().__init__()
            self.inner = inner

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.inner(x).unsqueeze(-1)


    def load_mlp_plr_replica(bundle_path, device):
        bundle = joblib.load(bundle_path)
        model  = MLPPLR(**bundle['arch_config'])
        model.load_state_dict(bundle['state_dict'])
        model.to(device).eval()
        return model, bundle['scaler']


    def scale_numeric_inplace(X_in: np.ndarray, scaler, num_idx) -> np.ndarray:
        out = X_in.copy()
        out[:, num_idx] = scaler.transform(X_in[:, num_idx])
        return out


    print('MLP-PLR setup complete (classes + helpers defined).')

## SHAP computation loop

For each pair and each replica:
1. Load the model (XGBoost was trained on unscaled data — no scaler needed here).
2. Compute TreeSHAP values via `shap.TreeExplainer` on the raw (unscaled) flagged instances.
3. Capture `explainer.expected_value` (log-odds base value) per replica.

Results are saved immediately after each pair to allow resuming.

In [ ]:
if MODEL_TYPE == 'xgboost':
    for p in pairs:
        pid       = p['pair_id']
        pair_dir  = MODEL_DIR / f'pair_{pid:02d}'
        shap_dir  = SHAP_DIR  / f'pair_{pid:02d}'

        shap_A_path = shap_dir / 'shap_A.npy'
        shap_B_path = shap_dir / 'shap_B.npy'

        if shap_A_path.exists() and shap_B_path.exists():
            print(f'Pair {pid:02d}: SHAP already computed, skipping.')
            continue

        print(f'\n── Pair {pid:02d} ──────────────────────────────────────────────')
        shap_dir.mkdir(parents=True, exist_ok=True)

        pred_data = np.load(pair_dir / 'predictions.npz')
        flagged_local_idx = pred_data['flagged_idx']
        idx_eval = np.array(p['idx_eval'], dtype=np.int64)

        flagged_global_idx = idx_eval[flagged_local_idx]
        X_flagged = X[flagged_global_idx]   # raw — XGBoost was trained on unscaled data

        n_flagged = X_flagged.shape[0]
        n_feat    = X_flagged.shape[1]
        print(f'  Flagged instances: {n_flagged:,}  Features: {n_feat}')

        if n_flagged == 0:
            print('  WARNING: no flagged instances — skipping.')
            continue

        # ── SHAP for replicas of window A ─────────────────────────────────
        shap_A = np.zeros((R, n_flagged, n_feat), dtype=np.float32)
        ev_A   = np.zeros(R, dtype=np.float32)
        for r in range(R):
            model = joblib.load(pair_dir / 'replicas_A' / f'model_r{r}.joblib')
            explainer = shap.TreeExplainer(model)
            vals = explainer.shap_values(X_flagged)
            if isinstance(vals, list):
                vals = vals[1]
            shap_A[r] = vals.astype(np.float32)
            raw_ev = explainer.expected_value
            ev_A[r] = np.float32(raw_ev if not isinstance(raw_ev, list) else raw_ev[1])
            print(f'  A replica {r}: SHAP computed  |mean|={np.abs(shap_A[r]).mean():.5f}  base={ev_A[r]:.4f}')

        # ── SHAP for replicas of window B ─────────────────────────────────
        shap_B = np.zeros((R, n_flagged, n_feat), dtype=np.float32)
        ev_B   = np.zeros(R, dtype=np.float32)
        for r in range(R):
            model = joblib.load(pair_dir / 'replicas_B' / f'model_r{r}.joblib')
            explainer = shap.TreeExplainer(model)
            vals = explainer.shap_values(X_flagged)
            if isinstance(vals, list):
                vals = vals[1]
            shap_B[r] = vals.astype(np.float32)
            raw_ev = explainer.expected_value
            ev_B[r] = np.float32(raw_ev if not isinstance(raw_ev, list) else raw_ev[1])
            print(f'  B replica {r}: SHAP computed  |mean|={np.abs(shap_B[r]).mean():.5f}  base={ev_B[r]:.4f}')

        # ── Sanity check: SHAP values should sum to log-odds prediction − base_value ──
        model_A0  = joblib.load(pair_dir / 'replicas_A' / 'model_r0.joblib')
        preds_raw = model_A0.predict(X_flagged, output_margin=True)
        shap_sum  = shap_A[0].sum(axis=1) + ev_A[0]
        max_err   = np.abs(shap_sum - preds_raw).max()
        print(f'  Sanity (A r0): max |SHAP_sum - log_odds| = {max_err:.6f}')
        if max_err > 0.01:
            print('  WARNING: SHAP consistency error too large!')
        else:
            print('  Sanity check passed.')

        np.save(shap_A_path, shap_A)
        np.save(shap_B_path, shap_B)
        np.save(shap_dir / 'expected_values_A.npy', ev_A)
        np.save(shap_dir / 'expected_values_B.npy', ev_B)
        print(f'  Saved shap_A {shap_A.shape}, shap_B {shap_B.shape}, expected_values A/B (R={R})')

    print('\n✓ All SHAP values computed.')

elif MODEL_TYPE == 'mlp_plr':
    BG_SIZE    = 100
    BATCH_SIZE = 512

    for p in pairs:
        pid       = p['pair_id']
        pair_dir  = MODEL_DIR / f'pair_{pid:02d}'
        shap_dir  = SHAP_DIR  / f'pair_{pid:02d}'

        shap_A_path = shap_dir / 'shap_A.npy'
        shap_B_path = shap_dir / 'shap_B.npy'

        if shap_A_path.exists() and shap_B_path.exists():
            print(f'Pair {pid:02d}: SHAP already computed, skipping.')
            continue

        print(f'\n── Pair {pid:02d} [mlp_plr] ──────────────────────────────────────')
        shap_dir.mkdir(parents=True, exist_ok=True)

        pred_data = np.load(pair_dir / 'predictions.npz')
        flagged_local_idx  = pred_data['flagged_idx']
        idx_A              = np.array(p['idx_A'],    dtype=np.int64)
        idx_B              = np.array(p['idx_B'],    dtype=np.int64)
        idx_eval           = np.array(p['idx_eval'], dtype=np.int64)
        flagged_global_idx = idx_eval[flagged_local_idx]
        X_flagged          = X[flagged_global_idx]

        n_flagged = X_flagged.shape[0]
        n_feat    = X_flagged.shape[1]
        print(f'  Flagged instances: {n_flagged:,}  Features: {n_feat}')

        if n_flagged == 0:
            print('  WARNING: no flagged instances — skipping.')
            continue

        shap_A = np.zeros((R, n_flagged, n_feat), dtype=np.float32)
        ev_A   = np.zeros(R, dtype=np.float32)
        shap_B = np.zeros((R, n_flagged, n_feat), dtype=np.float32)
        ev_B   = np.zeros(R, dtype=np.float32)

        for AB, idx_window, shap_out, ev_out in [
            ('A', idx_A, shap_A, ev_A),
            ('B', idx_B, shap_B, ev_B),
        ]:
            for r in range(R):
                bundle_path = pair_dir / f'replicas_{AB}' / f'model_r{r}.joblib'
                model, scaler = load_mlp_plr_replica(bundle_path, DEVICE)

                rng_bg = np.random.default_rng(r)
                n_bg   = min(BG_SIZE, len(idx_window))
                idx_bg = rng_bg.choice(idx_window, size=n_bg, replace=False)
                X_bg   = scale_numeric_inplace(X[idx_bg], scaler, num_col_idx)
                bg_t   = torch.from_numpy(X_bg).float().to(DEVICE)

                X_fl_s = scale_numeric_inplace(X_flagged, scaler, num_col_idx)
                fl_t   = torch.from_numpy(X_fl_s).float().to(DEVICE)

                model_shap = ShapOutputAdapter(model).to(DEVICE).eval()
                explainer  = shap.GradientExplainer(model_shap, bg_t)
                shap_batches = []
                for start in range(0, n_flagged, BATCH_SIZE):
                    chunk = fl_t[start:start + BATCH_SIZE]
                    vals  = explainer.shap_values(chunk)
                    if isinstance(vals, list):
                        vals = vals[0]
                    if isinstance(vals, np.ndarray) and vals.ndim == 3 and vals.shape[-1] == 1:
                        vals = vals[..., 0]
                    shap_batches.append(vals)
                shap_out[r] = np.concatenate(shap_batches, axis=0).astype(np.float32)

                with torch.no_grad():
                    ev_out[r] = float(model(bg_t).mean().item())

                mean_abs = float(np.abs(shap_out[r]).mean())
                print(f'  {AB} replica {r}: SHAP computed  |mean|={mean_abs:.5f}  base={ev_out[r]:.4f}')

                del model, model_shap, explainer, bg_t, fl_t
                torch.cuda.empty_cache()

        np.save(shap_A_path, shap_A)
        np.save(shap_B_path, shap_B)
        np.save(shap_dir / 'expected_values_A.npy', ev_A)
        np.save(shap_dir / 'expected_values_B.npy', ev_B)
        print(f'  Saved shap_A {shap_A.shape}, shap_B {shap_B.shape}, expected_values A/B (R={R})')

    print('\n✓ All SHAP values computed.')

elif MODEL_TYPE == 'logreg':
    print(f'MODEL_TYPE is "{MODEL_TYPE}" — SHAP computation skipped.')
    print('Logistic Regression uses its coefficients as the explanation; no post-hoc XAI needed.')

else:
    raise ValueError(f'Unknown MODEL_TYPE: {MODEL_TYPE}')

## SHAP stochasticity diagnostic (§3.9)

TreeSHAP is deterministic — running the explainer twice on the same model and data returns identical values. We verify this **for every pair** using the best-performing A replica (selected by PR-AUC from `per_replica_pr_auc_A`), and save the result as `stochasticity.json` per pair for use in notebook 04.

In [ ]:
if MODEL_TYPE == 'xgboost':
    for p in pairs:
        pid      = p['pair_id']
        pair_dir = MODEL_DIR / f'pair_{pid:02d}'
        shap_dir = SHAP_DIR  / f'pair_{pid:02d}'
        shap_dir.mkdir(parents=True, exist_ok=True)

        print(f'\n── Pair {pid:02d} stochasticity diagnostic ──')

        pred_data = np.load(pair_dir / 'predictions.npz')
        idx_eval  = np.array(p['idx_eval'], dtype=np.int64)
        flagged_local_idx  = pred_data['flagged_idx']
        flagged_global_idx = idx_eval[flagged_local_idx]
        X_flagged = X[flagged_global_idx]

        per_auc_A = pred_data['per_replica_pr_auc_A']
        best_r    = int(np.argmax(per_auc_A))
        print(f'  Best replica of A: r={best_r}  (PR-AUC={per_auc_A[best_r]:.4f})')

        model_best = joblib.load(pair_dir / 'replicas_A' / f'model_r{best_r}.joblib')
        exp = shap.TreeExplainer(model_best)

        run1 = exp.shap_values(X_flagged)
        if isinstance(run1, list): run1 = run1[1]
        run2 = exp.shap_values(X_flagged)
        if isinstance(run2, list): run2 = run2[1]

        max_abs_diff = float(np.abs(run1 - run2).max())
        is_det = max_abs_diff < 1e-6
        print(f'  Max |run1 - run2|: {max_abs_diff:.2e}')
        print('  TreeSHAP is deterministic ✓' if is_det else '  WARNING: non-zero stochasticity!')

        stoch = {
            'best_replica_A':   best_r,
            'max_abs_diff':     max_abs_diff,
            'is_deterministic': is_det,
        }
        with open(shap_dir / 'stochasticity.json', 'w') as f:
            json.dump(stoch, f, indent=2)
        print(f'  Saved stochasticity.json')

    print('\n✓ Stochasticity diagnostic complete.')

elif MODEL_TYPE == 'mlp_plr':
    BG_SIZE = 100

    for p in pairs:
        pid      = p['pair_id']
        pair_dir = MODEL_DIR / f'pair_{pid:02d}'
        shap_dir = SHAP_DIR  / f'pair_{pid:02d}'
        shap_dir.mkdir(parents=True, exist_ok=True)

        print(f'\n── Pair {pid:02d} [mlp_plr] stochasticity diagnostic ──')

        pred_data = np.load(pair_dir / 'predictions.npz')
        idx_A     = np.array(p['idx_A'],    dtype=np.int64)
        idx_eval  = np.array(p['idx_eval'], dtype=np.int64)
        flagged_local_idx  = pred_data['flagged_idx']
        flagged_global_idx = idx_eval[flagged_local_idx]
        X_flagged = X[flagged_global_idx]

        per_auc_A = pred_data['per_replica_pr_auc_A']
        best_r    = int(np.argmax(per_auc_A))
        print(f'  Best replica of A: r={best_r}  (PR-AUC={per_auc_A[best_r]:.4f})')

        model, scaler = load_mlp_plr_replica(
            pair_dir / 'replicas_A' / f'model_r{best_r}.joblib', DEVICE)

        rng_bg = np.random.default_rng(best_r)
        n_bg   = min(BG_SIZE, len(idx_A))
        idx_bg = rng_bg.choice(idx_A, size=n_bg, replace=False)
        X_bg   = scale_numeric_inplace(X[idx_bg], scaler, num_col_idx)
        bg_t   = torch.from_numpy(X_bg).float().to(DEVICE)

        X_fl_s = scale_numeric_inplace(X_flagged, scaler, num_col_idx)
        fl_t   = torch.from_numpy(X_fl_s).float().to(DEVICE)

        model_shap = ShapOutputAdapter(model).to(DEVICE).eval()
        explainer  = shap.GradientExplainer(model_shap, bg_t)

        np.random.seed(0)
        run1 = explainer.shap_values(fl_t)
        if isinstance(run1, list): run1 = run1[0]
        if isinstance(run1, np.ndarray) and run1.ndim == 3 and run1.shape[-1] == 1:
            run1 = run1[..., 0]

        np.random.seed(1)
        run2 = explainer.shap_values(fl_t)
        if isinstance(run2, list): run2 = run2[0]
        if isinstance(run2, np.ndarray) and run2.ndim == 3 and run2.shape[-1] == 1:
            run2 = run2[..., 0]

        max_abs_diff    = float(np.abs(run1 - run2).max())
        median_abs_diff = float(np.median(np.abs(run1 - run2)))
        is_det = max_abs_diff < 1e-6

        print(f'  Max    |run1 - run2|: {max_abs_diff:.2e}')
        print(f'  Median |run1 - run2|: {median_abs_diff:.2e}')
        print('  GradientExplainer is deterministic ✓' if is_det
              else '  GradientExplainer is stochastic (expected) ✓')

        stoch = {
            'best_replica_A':   best_r,
            'max_abs_diff':     max_abs_diff,
            'median_abs_diff':  median_abs_diff,
            'is_deterministic': is_det,
        }
        with open(shap_dir / 'stochasticity.json', 'w') as f:
            json.dump(stoch, f, indent=2)
        print(f'  Saved stochasticity.json')

        del model, model_shap, explainer, bg_t, fl_t
        torch.cuda.empty_cache()

    print('\n✓ Stochasticity diagnostic complete.')

## Quick SHAP summary (pair 0)

In [ ]:
if MODEL_TYPE in ('xgboost', 'mlp_plr'):
    import matplotlib.pyplot as plt

    shap_A_0 = np.load(SHAP_DIR / 'pair_00' / 'shap_A.npy')

    phi_bar_A = shap_A_0.mean(axis=0)

    global_imp = np.abs(phi_bar_A).mean(axis=0)
    top_k = 20
    top_idx = np.argsort(global_imp)[::-1][:top_k]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.barh(
        [feature_names[i] for i in top_idx[::-1]],
        global_imp[top_idx[::-1]],
        color='steelblue'
    )
    ax.set_title(f'Top {top_k} global SHAP importances (pair 0, model A, {MODEL_TYPE})')
    ax.set_xlabel('Mean |SHAP value|')
    plt.tight_layout()
    plt.savefig(SHAP_DIR / 'global_importance_pair00_A.png', dpi=120)
    plt.show()